# B3A — Dynamic vs precomputed-volume Tiling128 (CPU)

Chọn **Accelerator: None**, thêm dataset **V3 precomputed đủ199 CT/GT** bằng Add Input,
rồi Run All. Source đầy đủ của snapshot được nhúng trong notebook: không clone Git,
không cần push code, không có CT/GT/checkpoint bên trong notebook.

Chỉ export13 TRAIN pilot, so correctness rồi đo DataLoader CPU với cùng full keys,
2orders, workers0/2, batch2 và3 repetitions sau warmup. Memory gate có thể bỏ setup2-worker
nếu RAM không đủ; báo rõ NOT_RUN. Không model/train, không GPU/VRAM, không B3B/B4/B5.

V3 dùng `native` pass-through, không chuẩn hóa lần nữa. Output mới mỗi lần chạy;
giữ giới hạn provenance, không tuyên bố199 ca raw-exact.


In [ ]:
from pathlib import Path
import os, sys, subprocess, uuid
os.environ['CUDA_VISIBLE_DEVICES'] = ''
BASE = Path('/kaggle/working') / ('b3a_' + uuid.uuid4().hex[:8])
BASE.mkdir(parents=True, exist_ok=False)
PROJECT = BASE / 'project'
DATA_ROOT = None  # nếu có nhiều dataset V3, điền Path('/kaggle/input/...')
if DATA_ROOT is None:
    candidates = [p.parent for p in Path('/kaggle/input').rglob('merit_precomputed_v3.yaml')
                  if all((p.parent / n).is_file() for n in ('cases.json', 'splits_covid19_20_5fold.json', 'EXPORT_COMPLETE.json'))]
    if len(candidates) != 1:
        raise RuntimeError('Can chon dung DATA_ROOT V3; candidates=' + str(candidates))
    DATA_ROOT = candidates[0]
DATA_ROOT = Path(DATA_ROOT)
print('Data:', DATA_ROOT, '\nOutput:', BASE)


## 1. Giải nén và kiểm checksum source đã nhúng

In [ ]:
import base64, hashlib, io, json, zipfile
SOURCE_ZIP_B64 = '...'
EXPECTED_SOURCE_SHA256 = 'c0b71a6fe4c974992209a80482c899f8554dc2c7648e8f5a97d0af97622bd190'
payload = base64.b64decode(SOURCE_ZIP_B64)
assert hashlib.sha256(payload).hexdigest() == EXPECTED_SOURCE_SHA256
PROJECT.mkdir()
with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    assert archive.testzip() is None
    for name in archive.namelist():
        assert (PROJECT / name).resolve().is_relative_to(PROJECT.resolve())
    archive.extractall(PROJECT)
for name, expected in json.loads((PROJECT / 'SOURCE_SNAPSHOT.json').read_text()).items():
    assert hashlib.sha256((PROJECT / name).read_bytes()).hexdigest() == expected, name
print('Source SHA PASS; files:', len(json.loads((PROJECT / 'SOURCE_SNAPSHOT.json').read_text())))


## 2. Dependency và test synthetic trước khi đọc pilot thật

In [ ]:
import importlib.util
small_dependencies = {'numpy':'numpy>=1.24', 'nibabel':'nibabel>=5', 'yaml':'PyYAML>=6',
                      'psutil':'psutil>=5', 'pytest':'pytest>=7'}
missing = [pkg for module, pkg in small_dependencies.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', *missing], check=True)
if importlib.util.find_spec('torch') is None:
    raise RuntimeError('Can cai Torch CPU truoc; notebook khong tu tai bo Torch/CUDA lon.')
XML = BASE / 'synthetic.xml'
test = subprocess.run([sys.executable, '-m', 'pytest', 'tests/test_b3a_delivery.py', '-q',
    '--basetemp', str(BASE / 'pytest_tmp'), '--junitxml', str(XML)], cwd=PROJECT,
    text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
(BASE / 'synthetic.log').write_text(test.stdout, encoding='utf-8')
print(test.stdout)
test.check_returncode()


## 3. Stage cùng storage, export subset, correctness A-0003 rồi benchmark

Lượt này có thể mất nhiều phút. Log mỗi2000 batch; không phải train. Không dừng trước correctness để tự lấy số benchmark.

In [ ]:
import json
EXPECTED_PILOT = json.loads('{"case_ids": ["volume-covid19-A-0003", "volume-covid19-A-0698", "volume-covid19-A-0413", "volume-covid19-A-0511", "volume-covid19-A-0502", "volume-covid19-A-0031", "volume-covid19-A-0320", "volume-covid19-A-0288", "volume-covid19-A-0016", "volume-covid19-A-0215", "volume-covid19-A-0557", "volume-covid19-A-0013", "volume-covid19-A-0473"], "stats_sha256": "583b32eb4bdeeef36db9dbc10282b221b13cd8add18c7ff18230e7f5aa9a38c4", "parameters": {"lower": -1241.0, "upper": 1032.0, "mean": -583.0622212292245, "std": 488.0394718302036}, "input_hashes": {"volume-covid19-A-0003": {"ct": "f3f9b1f31720fd2da8fab5fa165e292659e177d1f7df7dd225d94e8436faf7ba", "gt": "7b616bf123318642333bf74df0529977f53df935889faf40b39263db73909652"}, "volume-covid19-A-0698": {"ct": "000eff200d143d527e8a5e185123334e948c36edde9af76508f508c184a160de", "gt": "38804fef77f958a30b9628abdde5220dfc7a26823938decdc2d62f3d04aca743"}, "volume-covid19-A-0413": {"ct": "a0784857a4a76edd491a4dff670a386d7b5e295738233748f5774ff2a1390b33", "gt": "da7a6fe87fe8d5cfe080f65f180237aa090341d0b9c83acb647a3f3cf4da73d6"}, "volume-covid19-A-0511": {"ct": "a4a39c032fbf6e7e5b32c18f073aed76018c232a949fcb245eee04d3af321031", "gt": "b371ac1967aed30953b695baa3edf323524636ffef1003a56348b3c6b6f112ea"}, "volume-covid19-A-0502": {"ct": "be381cd98319b8b0e6eff06538a68cc8340a8918f945605314d5755fa488789c", "gt": "bc34abea41957334df4d321fd4edfeac69b63fa12989976801fc599bdfca8623"}, "volume-covid19-A-0031": {"ct": "2e217b0dc7a80b5eab5dc9b2be6ab7883800d829d1b3bc2e7334c63882adc354", "gt": "83dcf2c4ff203717791b684269d34a9b7f97ac58e3c6fcd57d67ae0a76d77785"}, "volume-covid19-A-0320": {"ct": "aef9397649b6e0df3783d1fd97edd1956da2375fc79450eaa35d464184d7cecc", "gt": "dd356a99fccd69af1ecd47bacd68c8a69bfd6e7e3c59b5f99abd2fd0ec2eb09f"}, "volume-covid19-A-0288": {"ct": "97b9adc2794691260d42402828ffd1a18db5ed071c6bb9b7550b49bd0fda7f8a", "gt": "8edd3995e1558696d1e1658e2deb7a3f9ac4980408438ae7d31fa7b17c780d60"}, "volume-covid19-A-0016": {"ct": "6c06dd12c289146a20fef87a9c2b961216d30ab83f90cc93e7e2f8a9009ea563", "gt": "9e5c5a5c7758873ff9b434dde8e88ac670c816453de69871935155d5e19d7b46"}, "volume-covid19-A-0215": {"ct": "0d9241b3bfa02ecdde582d26fbef56c74cd4ec555c9bdde809edaee8922f040e", "gt": "3e3a59b8f260732ad569e49a4ab7f9bdb259e790195ceb13a4fa5a5208b06fb8"}, "volume-covid19-A-0557": {"ct": "10e2f2ae673ad12547cd0a721d4ec6fdaf6ae8e9d8fdb6e781baac65fc1b847a", "gt": "4cf474f2471d366530834f6baf5eaec223e83dbd12ed741613c3bccfbe3e6dc9"}, "volume-covid19-A-0013": {"ct": "0c8f7ca31d690b8fd28b28baefadf2a0d262c996577dfe10d629f75a8a2dc3f6", "gt": "7caa7685031fc796e2389dfb83157b814187f0929f386a579270ada1186e7264"}, "volume-covid19-A-0473": {"ct": "69c45cad3c34ed9717ebfe6f0ab60b34c52b65b609ffde490cabc13512870f44", "gt": "376e250b95c21fca52162b5181a0fb64c5f70c7afc9c9ea007a787cf3de26376"}}}')
sys.path[:0] = [str(PROJECT), str(PROJECT / 'src')]
from tools.qc_tiling128 import load_inputs
from tools.b3a_compare import select_pilot
cases, parts, config, exported, entries, _, _ = load_inputs(DATA_ROOT, 'cases.json', 'splits_covid19_20_5fold.json', 'merit_precomputed_v3.yaml', 'EXPORT_COMPLETE.json')
chosen = select_pilot(cases, entries, parts)
assert [c.case_id for c in chosen] == EXPECTED_PILOT['case_ids'], 'Pilot subset differs from reviewed local experiment'
assert exported['stats_sha256'] == EXPECTED_PILOT['stats_sha256'] and exported['parameters'] == EXPECTED_PILOT['parameters'], 'Different V3 stats'
for name, expected in EXPECTED_PILOT['input_hashes'].items():
    assert entries[name]['ct_output_sha256'] == expected['ct'] and entries[name]['gt_output_sha256'] == expected['gt'], name
print('Pinned13-case subset and V3 metadata hashes match local reference.')
EXPERIMENT = BASE / 'experiment'
command = [sys.executable, '-u', 'tools/b3a_compare.py', '--data-root', str(DATA_ROOT),
    '--output-dir', str(EXPERIMENT), '--synthetic-xml', str(XML),
    '--workers', '0', '2', '--batch-size', '2', '--repetitions', '3']
with (BASE / 'benchmark.log').open('w', encoding='utf-8') as log:
    process = subprocess.Popen(command, cwd=PROJECT, text=True, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, bufsize=1)
    for line in process.stdout:
        print(line, end='')
        log.write(line); log.flush()
    if process.wait() != 0:
        raise RuntimeError('B3A STOP; xem benchmark.log/B3A_ERROR.json, khong tiep tuc model/train.')


## 4. Báo cáo và ZIP nhỏ để gửi review

In [ ]:
REPORT = BASE / 'report'
subprocess.run([sys.executable, 'tools/b3a_reports.py', '--experiment', str(EXPERIMENT),
                '--output-dir', str(REPORT)], cwd=PROJECT, check=True)
print((REPORT / 'b3a_decision.md').read_text(encoding='utf-8'))
import zipfile
REPORT_ZIP = BASE / 'B3A_KAGGLE_REPORT.zip'
with zipfile.ZipFile(REPORT_ZIP, 'x', zipfile.ZIP_DEFLATED) as archive:
    for p in EXPERIMENT.iterdir():
        if p.is_file() and p.suffix in ('.json', '.csv'):
            archive.write(p, 'reports/' + p.name)
    for p in REPORT.iterdir():
        if p.is_file(): archive.write(p, p.name)
    for p in (XML, BASE/'synthetic.log', BASE/'benchmark.log'):
        archive.write(p, 'evidence/' + p.name)
    archive.write(PROJECT/'SOURCE_SNAPSHOT.json', 'SOURCE_SNAPSHOT.json')
print('Tai ZIP tu Output Files:', REPORT_ZIP)
print('Da dung B3A; khong B3B/model/train/GPU benchmark.')
